# Evaluate the trained ball detectors (WASB fine-tuning)

Change `train-ball-detector`. The training notebook (`train_ball_detector.ipynb`) trained both runs but hit Kaggle's
12-hour limit before its evaluation finished, and its onnxruntime-gpu (built for CUDA 13) fell back to the CPU. This
notebook evaluates the published WASB weights and both runs (`gym`, `mixed`) on the held-out broadcast and gym matches
with `scripts/eval_ball_detection.py`. Inputs: the training kernel's output (`yoga0702/train-ball-detector`),
`sherif31/group-activity-recognition-volleyball`, `yoga0702/vball-orel` (private; research use).

In [ ]:
import glob, os, shutil
WORK = '/tmp/work'; OUT = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)
VIDEOS = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/videos/47', recursive=True))
OREL = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/test/match1', recursive=True)[0]))  # holds train/ and test/
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q --depth 1 https://github.com/nttcom/WASB-SBDT {WORK}/wasb
!git clone -q --depth 1 https://github.com/DL-Volleyball-Analysis/volleyball-analysis {WORK}/repo
!git clone -q --depth 1 https://github.com/asigatchov/fast-volleyball-tracking-inference {WORK}/repo/external/fast-volleyball-tracking-inference
!mkdir -p {WORK}/repo/models && cp {WORK}/repo/external/fast-volleyball-tracking-inference/models/VballNetV4c_seq9_grayscale_*.onnx {WORK}/repo/models/vballnet_v4c.onnx
!pip -q install -e {WORK}/repo --no-deps --ignore-requires-python  # Kaggle's Python may be older than the repo's 3.12 floor
!pip -q install onnx 'onnxruntime-gpu==1.22.0' gdown omegaconf  # 1.22 is built for CUDA 12, which Kaggle has
!gdown -q 1M9y4wPJqLc0K-z-Bo5DP8Ft5XwJuLqIS -O {WORK}/wasb_volleyball_best.pth.tar
!gdown -q 1urZpZiiepC85JD1u3VeURgUpztRgI0yl -O {WORK}/ball_ann.zip && unzip -q -o {WORK}/ball_ann.zip -d {WORK}/ann
ANN = f'{WORK}/ann/volleyball_ball_annotation'
import onnxruntime as ort
print(ort.__version__, ort.get_available_providers())
RUNS = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/runs/wasb_published.onnx', recursive=True))


In [ ]:
for name, onnx in (('published', f'{RUNS}/wasb_published.onnx'), ('gym', f'{RUNS}/gym/best.onnx'), ('mixed', f'{RUNS}/mixed/best.onnx')):
    shutil.copy(onnx, f'{WORK}/repo/models/wasb_volleyball.onnx')
    !cd {WORK}/repo && python scripts/eval_ball_detection.py --videos {VIDEOS} --ann {ANN} --detectors wasb --out {OUT}/eval/{name}_broadcast
    !cd {WORK}/repo && python scripts/eval_ball_detection.py --dataset orel --orel {OREL} --detectors wasb --out {OUT}/eval/{name}_gym
# VballNet (the pipeline's current detector) on the same gym test matches, for the switch decision
!cd {WORK}/repo && python scripts/eval_ball_detection.py --dataset orel --orel {OREL} --detectors v4c --out {OUT}/eval/vballnet_gym
